# Module 06 — EDA Interactive Notebook
## The Darko Method 2026 | GlobalTech Solutions Workforce Analysis

---

### How to use this notebook
Run cells from top to bottom using **Shift+Enter**.
Each cell has a comment explaining what it does and why.

This notebook is the **interactive layer** on top of `run.py`.
While `run.py` produces text reports, this notebook produces:
- Visual charts (histograms, box plots, scatter plots, heatmaps)
- Interactive exploration of specific findings
- Deeper investigation of anomalies

**Business questions we answer here:**
1. How is salary distributed across the company?
2. Which departments are outliers in pay or performance?
3. Does experience predict salary as expected?
4. Which specific employees are statistical anomalies?


In [ ]:
# ── Cell 1: Imports and Setup ─────────────────────────────────────────────
# Before anything else, we import the libraries we need.
# These are the standard imports for any EDA notebook.

import sys
import pathlib

# Add the project root to Python path so we can import our classes
# pathlib.Path.cwd() returns the current working directory
# We walk up until we find config.py
_root = pathlib.Path.cwd()
while not (_root / 'config.py').exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt   # matplotlib: the core Python plotting library
import seaborn as sns             # seaborn: statistical plots built on matplotlib

# Tell matplotlib to display plots directly in the notebook (not in a separate window)
%matplotlib inline

# Set seaborn style: 'whitegrid' adds a subtle grid which makes values easier to read
sns.set_style('whitegrid')

# Set figure size default: (width_inches, height_inches)
plt.rcParams['figure.figsize'] = (12, 5)

# Import our pipeline classes
from config import DATA_PATH, REPORTS_DIR, INDUSTRY
from src.eda_engine import EDAEngine
from src.anomaly_detector import AnomalyDetector

print(f'Setup complete. Industry: {INDUSTRY}')
print(f'Data file: {DATA_PATH}')

In [ ]:
# ── Cell 2: Load and Profile the Data ────────────────────────────────────
# We use our EDAEngine class to load and profile the data.
# This is the same code that run.py executes — but here we can
# inspect intermediate results interactively.

engine = EDAEngine()
engine.load().profile()

# Access the profile results dictionary directly
p = engine.results['profile']

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Rows:          {p["rows"]:,}')
print(f'  Columns:       {p["columns"]}')
print(f'  Numeric cols:  {p["numeric_cols"]}')
print(f'  Text cols:     {p["categorical_cols"]}')
print(f'  Missing vals:  {p["total_nulls"]:,} ({p["null_pct"]}%)')
print(f'  Memory:        {p["memory_mb"]} MB')

In [ ]:
# ── Cell 3: Descriptive Statistics ───────────────────────────────────────
# pandas .describe() gives us the 8 key statistics for every numeric column.
# We display it as a formatted table directly in the notebook.

# Select only columns that are not pipeline metadata (starting with _)
analysis_df = engine.df.drop(
    columns=[c for c in engine.df.columns if c.startswith('_')],
    errors='ignore'
)

# .describe() → count, mean, std, min, 25%, 50%, 75%, max for each numeric col
# .T transposes the table: columns become rows (easier to read with many columns)
# .style.background_gradient() adds colour — higher values are darker blue
desc = analysis_df.describe().round(2).T
desc

In [ ]:
# ── Cell 4: Salary Distribution — Histogram ───────────────────────────────
# A histogram shows how values are distributed across ranges.
# BINS divide the value range into equal segments and count how many
# values fall in each bin.
#
# What to look for:
#   Normal (bell curve) → values cluster symmetrically around the mean
#   Right skewed        → most values are low, a few are very high (common for income)
#   Bimodal (two peaks) → two distinct groups (e.g. junior vs senior employees)

if 'salary' in analysis_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left plot: histogram with KDE curve
    # kde=True adds a smoothed density curve on top of the bars
    sns.histplot(analysis_df['salary'].dropna(), bins=30, kde=True,
                 ax=axes[0], color='steelblue')
    axes[0].set_title('Salary Distribution', fontsize=13)
    axes[0].set_xlabel('Salary (£)')
    axes[0].set_ylabel('Employee Count')
    # Add vertical lines for mean and median to show the difference
    axes[0].axvline(analysis_df['salary'].mean(),   color='red',    linestyle='--', label='Mean')
    axes[0].axvline(analysis_df['salary'].median(), color='orange', linestyle='--', label='Median')
    axes[0].legend()

    # Right plot: box plot shows quartiles and outliers visually
    # The box shows Q1 to Q3. The whiskers extend to the IQR fences.
    # Dots beyond the whiskers are the outliers IQR already identified.
    if 'department' in analysis_df.columns:
        sns.boxplot(data=analysis_df, x='department', y='salary',
                    ax=axes[1], palette='Set2')
        axes[1].set_title('Salary by Department', fontsize=13)
        axes[1].tick_params(axis='x', rotation=45)
    else:
        sns.boxplot(y=analysis_df['salary'].dropna(), ax=axes[1])
        axes[1].set_title('Salary Box Plot', fontsize=13)

    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'salary_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to reports/salary_distribution.png')
else:
    # If there is no salary column, plot the first numeric column instead
    col = engine.num_cols[0] if engine.num_cols else None
    if col:
        sns.histplot(analysis_df[col].dropna(), bins=30, kde=True, color='steelblue')
        plt.title(f'{col} Distribution')
        plt.show()

In [ ]:
# ── Cell 5: Group Analysis — Bar Chart ───────────────────────────────────
# Bar charts are the most intuitive way to compare group-level metrics.
# We show the mean of the first numeric column grouped by the first
# categorical column.

engine.group_analysis()

group_data = engine.results.get('group_analysis', {})

if group_data:
    first_cat = list(group_data.keys())[0]
    first_num = list(group_data[first_cat].keys())[0]
    rows = group_data[first_cat][first_num]

    df_group = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(12, 5))

    # .sort_values('mean') puts longest bar at top when we set orientation
    df_sorted = df_group.sort_values('mean', ascending=True)

    # barh = horizontal bar chart (easier to read long category names)
    bars = ax.barh(df_sorted[first_cat].astype(str),
                   df_sorted['mean'],
                   color=sns.color_palette('Blues_d', len(df_sorted)))

    # Add value labels at the end of each bar
    for bar, val in zip(bars, df_sorted['mean']):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f'{val:,.0f}', va='center', fontsize=10)

    ax.set_title(f'Mean {first_num.replace("_"," ").title()} by {first_cat.replace("_"," ").title()}',
                 fontsize=13)
    ax.set_xlabel(f'Mean {first_num}')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'group_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to reports/group_analysis.png')

In [ ]:
# ── Cell 6: Correlation Heatmap ───────────────────────────────────────────
# A heatmap visualises the entire correlation matrix at once.
# Each cell is coloured by the correlation value:
#   Dark blue → strong positive correlation (+1)
#   White     → no correlation (0)
#   Dark red  → strong negative correlation (-1)
#
# This is how data scientists quickly identify:
#   - Which features are strongly related to the target
#   - Which features are redundant (correlated with each other)

engine.correlation()

if 'correlation' in engine.results:
    # Rebuild the correlation matrix from the stored dict
    corr_df = pd.DataFrame(engine.results['correlation']['matrix'])

    fig, ax = plt.subplots(figsize=(10, 8))

    # sns.heatmap() colours each cell by its value
    # annot=True prints the number inside each cell
    # fmt='.2f' formats numbers to 2 decimal places
    # cmap='RdBu_r': red-blue colour map (reversed so positive = blue, negative = red)
    # vmin/vmax fix the colour scale from -1 to +1
    # square=True makes cells square (not rectangular)
    # linewidths adds thin lines between cells for readability
    sns.heatmap(
        corr_df,
        annot=True, fmt='.2f',
        cmap='RdBu_r', vmin=-1, vmax=1,
        square=True, linewidths=0.5,
        ax=ax
    )
    ax.set_title('Correlation Matrix — Numeric Features', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to reports/correlation_heatmap.png')

    print('\nStrongest correlations:')
    for pair in engine.results['correlation']['strong_pairs'][:5]:
        direction = '↑' if pair['direction'] == 'positive' else '↓'
        print(f"  {pair['col_a']} ↔ {pair['col_b']}: r={pair['correlation']:.3f} ({pair['strength']} {direction})")

In [ ]:
# ── Cell 7: Scatter Plot — Experience vs Salary ───────────────────────────
# A scatter plot shows the relationship between two numeric variables.
# Each dot = one employee. X position = experience. Y position = salary.
#
# The REGRESSION LINE (trend line) shows the linear relationship.
# If the line goes up left-to-right, experience and salary are positively correlated.
#
# Points FAR from the trend line are interesting:
#   High up, low experience → potential anomaly or high performer
#   Low down, high experience → possible underpayment or inactive role

num_cols = engine.num_cols

if len(num_cols) >= 2:
    x_col = num_cols[0]
    y_col = num_cols[1] if len(num_cols) > 1 else num_cols[0]

    fig, ax = plt.subplots(figsize=(10, 6))

    # If a categorical column exists, colour points by category
    if engine.cat_cols:
        hue_col = engine.cat_cols[0]
        analysis_df = engine.df.drop(
            columns=[c for c in engine.df.columns if c.startswith('_')], errors='ignore')
        sns.scatterplot(
            data=analysis_df, x=x_col, y=y_col,
            hue=hue_col, alpha=0.6, ax=ax, palette='tab10'
        )
    else:
        analysis_df = engine.df.drop(
            columns=[c for c in engine.df.columns if c.startswith('_')], errors='ignore')
        sns.scatterplot(data=analysis_df, x=x_col, y=y_col, alpha=0.6, ax=ax)

    # Add a regression (trend) line using regplot
    # scatter=False means draw only the trend line (no dots, we already drew them)
    # color='red' makes it stand out
    sns.regplot(
        data=analysis_df, x=x_col, y=y_col,
        scatter=False, color='red', line_kws={'linewidth': 1.5}, ax=ax
    )

    ax.set_title(f'{y_col.replace("_"," ").title()} vs {x_col.replace("_"," ").title()}',
                 fontsize=13)
    ax.set_xlabel(x_col.replace('_', ' ').title())
    ax.set_ylabel(y_col.replace('_', ' ').title())
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'scatter_relationship.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to reports/scatter_relationship.png')

In [ ]:
# ── Cell 8: Anomaly Detection ─────────────────────────────────────────────
# Now we run the AnomalyDetector on the loaded data.
# Confirmed anomalies are those flagged by BOTH IQR and Z-score.

detector = AnomalyDetector(engine.df)
detector.run()  # IQR + Z-score consensus on first 3 numeric columns

print('Anomaly Detection Summary:')
print(detector.summary().to_string(index=False))
print(f'\nTotal confirmed anomaly rows: {len(detector.confirmed):,}')

In [ ]:
# ── Cell 9: Visualise Anomalies ───────────────────────────────────────────
# Show which rows are anomalies on the scatter plot.
# Normal rows = small blue dots
# Anomaly rows = large red X marks

if len(detector.confirmed) > 0 and len(num_cols) >= 2:
    x_col = num_cols[0]
    y_col = num_cols[1]

    analysis_df = engine.df.drop(
        columns=[c for c in engine.df.columns if c.startswith('_')], errors='ignore')

    # Separate normal and anomalous rows
    normal_mask   = ~engine.df.index.isin(detector.confirmed.index)
    normal_rows   = analysis_df[normal_mask]
    anomaly_rows  = analysis_df[~normal_mask]

    fig, ax = plt.subplots(figsize=(11, 6))

    # Plot normal rows as small, transparent blue dots
    ax.scatter(normal_rows[x_col], normal_rows[y_col],
               c='steelblue', alpha=0.4, s=20, label='Normal')

    # Plot anomaly rows as large, bold red X markers
    ax.scatter(anomaly_rows[x_col], anomaly_rows[y_col],
               c='red', marker='X', s=150, label=f'Anomaly ({len(anomaly_rows)} rows)',
               edgecolors='darkred', linewidths=0.5)

    ax.set_title('Confirmed Anomalies (IQR + Z-score Consensus)', fontsize=13)
    ax.set_xlabel(x_col.replace('_', ' ').title())
    ax.set_ylabel(y_col.replace('_', ' ').title())
    ax.legend()
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'anomalies_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to reports/anomalies_plot.png')

    print('\nAnomaly rows (top 5):')
    print(anomaly_rows.head())
else:
    print('No confirmed anomalies found — data is clean!')

## Summary and Next Steps

You have completed the EDA for this dataset. Record your key findings below:

### Key Findings

| Finding | Value | Action |
|---|---|---|
| Top group by mean salary | *fill in* | |
| Strongest correlation | *fill in* | |
| Anomaly rows confirmed | *fill in* | Review anomalies.csv |

### Next Steps
- **Module 09 ML**: Use the strongest correlation features for model training
- **Module 11 LLM**: Pass the profile stats dict to Claude as context
- **Module 14 MLOps**: Save the profile as baseline for drift detection

### Questions for the VP of People
1. *(Write your first business insight here)*
2. *(Write your second business insight here)*
3. *(Write your recommendation here)*
